# 18 - ¿Cambia algo una pérdida que mire el borde?

## La pregunta

Luís, por Slack: *"o maior erro são nas bordas... daria para colocar uma loss composta"*.

La tesis ya sostiene las dos mitades del problema:

- **Sección 4.5**: entre el 80 % y el 90 % de los píxeles donde predicción y referencia
  discrepan caen en una banda de ±5 px alrededor del contorno, que es el 2,6 % de la imagen.
- **Línea 607**: la pérdida supervisada es `BCE + Dice`. **Las dos son de región**: tratan
  igual a un píxel del centro de la vértebra que a uno del borde.

O sea: el error vive en el borde y la pérdida no mira el borde. Este notebook prueba si
eso importa.

## El diseño

Un solo escalar cambia entre runs. Todo lo demás (arquitectura, aumentos, learning rate,
early stopping, semillas) es idéntico al supervisado de `runs_final_v1/supervised/`.

```
L = BCE + Dice + w * L_borde
```

Dos formulaciones de `L_borde`, ambas parametrizadas:

| modo | qué hace |
|---|---|
| `dw` | pondera la BCE por una gaussiana de la distancia al contorno de la referencia |
| `kervadec` | Boundary Loss (Kervadec 2019): integral de la probabilidad contra el mapa de distancia con signo |

Con `w = 0` la pérdida es **exactamente** la de siempre. Eso está comprobado con una
puerta de equivalencia numérica, y se vuelve a comprobar en el Paso 1 de este notebook.

## El coste

Medido: el supervisado converge en la época 15 y para en la 55 por early stopping.
Un run entero está en **15-20 min**. El término de borde añade ~35 ms por iteración
sobre 257 ms, o sea un 13 %.

```
Paso 2 (barrido)  2 modos x 4 pesos x 1 semilla =  8 runs  ~2,5 h
Paso 4 (final)    2 modos x 1 peso  x 3 semillas =  4 runs nuevos  ~1,3 h
                                                    -----------------------
                                                    ~4 h, cabe en una noche
```

## Lo que NO hace este notebook

No toca `runs_final_v1/` ni ningún resultado existente. Escribe solo en
`runs_boundary_loss/`. Y **no está pensado para entrar en la tesis por defecto**: el
destino natural del resultado es el banco de preguntas de defensa. Si sale espectacular,
ya se decidirá.

## El λ se elige en VALIDACIÓN

El Paso 3 elige el peso por `hd95_val` del `run_report`, **nunca por test**. Elegir el
hiperparámetro mirando el test invalidaría la comparación entera.


In [ ]:
# ============================================================
# SETUP - correr una vez tras cada reinicio del runtime
# No entrena nada.
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

!git clone https://github.com/sebastianquispearias/tesis-seg.git
%cd tesis-seg
!pip install -q -r requirements.txt

import torch, sys
print("Python    :", sys.version)
print("torch     :", torch.__version__)
print("CUDA      :", torch.version.cuda)
!git log --oneline -1
!nvidia-smi | grep -E "NVIDIA|Driver Version|CUDA Version"

import sys
sys.path.append("/content/tesis-seg")

import json, os, time, glob, statistics

from src.defaults import get_default_config, summarize_config
from src.augmentations import get_supervised_train_augmentation
from src.datasets import build_supervised_datasets, build_dataloaders
from src.train import run_training
from src.evaluate import evaluate_checkpoint

# --- GUARDIAN: el codigo clonado debe traer la perdida de borde ---
# El 2026-08-16 se perdieron 20 runs porque un flag se ignoro en silencio.
# Aqui el riesgo es identico: si el repo clonado es viejo, build_criterion
# lanzaria ValueError, pero si alguien lo "arregla" poniendo bce_dice se
# entrenarian 12 runs identicos al baseline sin que nada avise.
import inspect
from src import losses as _ls
_src = inspect.getsource(_ls)
assert "BCEPlusDicePlusBoundary" in _src, (
    "CODIGO VIEJO: src/losses.py no tiene BCEPlusDicePlusBoundary. "
    "Borra /content/tesis-seg, vuelve a clonar y reinicia el entorno.")
assert "boundary_weight" in inspect.getsource(_ls.build_criterion), (
    "CODIGO VIEJO: build_criterion no lee boundary_weight del config.")
print("OK: el codigo clonado tiene la perdida de borde y la lee del config.")


---
### Paso 1 - Verificar ANTES de gastar GPU

No entrena. Comprueba las tres cosas que, si fallan, harían inútil toda la noche:

1. **Puerta de equivalencia**: con `boundary_weight = 0` la pérdida nueva devuelve
   exactamente el mismo número que `BCEPlusDice`. Si no, el baseline no es comparable.
2. **Escalas**: los dos modos tienen que dar términos del mismo orden de magnitud, o el
   mismo barrido de `w` no sirve para los dos.
3. **El flag llega**: `build_criterion` construye la clase correcta desde el `cfg`.


In [ ]:
# ============================================================
# VERIFICACION - NO ENTRENA. Tarda ~1 minuto.
# ============================================================
from src.losses import BCEPlusDice, BCEPlusDicePlusBoundary, build_criterion

BASE = "/content/drive/MyDrive/UNM_vertebras_seg_v3"

_cfg = get_default_config()
_cfg["img_root"] = BASE
_cfg["msk_root"] = BASE
_cfg["num_workers"] = 0
_cfg["batch_size"] = 5
_cfg["use_semi"] = False

_tr, _va, _te = build_supervised_datasets(_cfg, train_tf=None)
_ld = build_dataloaders(_cfg, _tr, _va, _te)
_b = next(iter(_ld["train_loader"]))

_dev = "cuda" if torch.cuda.is_available() else "cpu"
_y = _b["mask"].to(_dev)
torch.manual_seed(0)
_logits = (torch.randn_like(_y) * 2.0).requires_grad_(True)
print("batch: image %s | mask %s | positivos %.3f%%" % (tuple(_b["image"].shape), tuple(_y.shape), 100 * float(_y.mean())))
print()

# --- 1. puerta de equivalencia ---
_v_old, _ = BCEPlusDice()(_logits, _y, epoch=3)
_v_new, _ = BCEPlusDicePlusBoundary(mode="dw", boundary_weight=0.0)(_logits, _y, epoch=3)
_diff = abs(float(_v_old) - float(_v_new))
print("PUERTA DE EQUIVALENCIA (boundary_weight = 0)")
print("  BCEPlusDice                  : %.12f" % float(_v_old))
print("  BCEPlusDicePlusBoundary(w=0) : %.12f" % float(_v_new))
print("  diferencia                   : %.3e" % _diff)
assert _diff < 1e-9, "FALLA: la perdida nueva con w=0 no reproduce la vieja. NO entrenar."
print("  OK\n")

# --- 2. escalas de los dos modos ---
print("ESCALA DEL TERMINO DE BORDE POR MODO")
_scales = {}
for _m in ("dw", "kervadec"):
    _, _c = BCEPlusDicePlusBoundary(mode=_m, boundary_weight=1.0)(_logits, _y, epoch=None)
    _scales[_m] = _c["boundary_loss"]
    print("  %-10s termino = %8.4f   (escala cruda del mapa: %.1f)" % (_m, _c["boundary_loss"], _c["boundary_raw_scale"]))
_ratio = max(_scales.values()) / max(1e-9, min(abs(v) for v in _scales.values()))
print("  razon entre modos: %.1fx" % _ratio)
assert _ratio < 20, ("Los dos modos estan en escalas muy distintas; un solo barrido de w "
                     "no serviria para ambos.")
print("  OK: el mismo barrido de w vale para los dos modos\n")

# --- 3. el flag llega desde el cfg ---
print("build_criterion DESDE EL CONFIG")
for _ln, _extra in (("bce_dice", {}),
                    ("bce_dice_boundary", {"boundary_mode": "dw", "boundary_weight": 1.0}),
                    ("bce_dice_boundary", {"boundary_mode": "kervadec", "boundary_weight": 1.0})):
    _c2 = dict(_cfg); _c2["loss_name"] = _ln; _c2.update(_extra)
    _obj = build_criterion(_c2)
    print("  %-18s %-46s -> %s" % (_ln, str(_extra), type(_obj).__name__))
    if _ln != "bce_dice":
        assert isinstance(_obj, BCEPlusDicePlusBoundary)
        assert _obj.mode == _extra["boundary_mode"]
print("  OK\n")
print("TODO VERIFICADO. Se puede entrenar.")


---
### Paso 1b - Los parámetros del experimento

**Todo lo que se puede tocar está en esta celda y solo en esta celda.** Si quieres
cambiar los modos, los pesos o las semillas, es aquí.


In [ ]:
# ============================================================
# PARAMETROS DEL EXPERIMENTO - lo unico que hay que tocar
# ============================================================
BASE     = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
ROTULOS  = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"
OUT_ROOT = f"{BASE}/runs_boundary_loss"          # directorio NUEVO, no toca nada

# --- las dos perdidas a probar. Quitar una de la lista para probar solo la otra.
MODOS = ["dw", "kervadec"]

# --- el peso del termino de borde. Es EL hiperparametro del experimento.
#     Los dos modos estan normalizados a la misma escala (verificado en el Paso 1),
#     asi que este mismo barrido vale para ambos.
PESOS_BARRIDO = [0.1, 0.3, 1.0, 3.0]

# --- semillas
SEMILLA_BARRIDO = 0            # el barrido usa una sola, para no gastar la noche
SEMILLAS_FINAL  = [0, 1, 2]    # el mejor peso de cada modo se repite con estas

# --- parametro de la gaussiana del modo "dw", en pixeles.
#     5 px es el mismo ancho de banda que usa la seccion 4.5 para medir el error.
DW_SIGMA = 5.0

# --- rampa del termino de borde. 0 = sin rampa, activo desde la primera epoca.
#     Ponerlo en (5, 10) si algun run diverge en las primeras epocas.
BOUNDARY_START_EPOCH   = 0
BOUNDARY_WARMUP_EPOCHS = 0

# --- el baseline contra el que se compara. NO se reentrena: ya existe.
BASELINE_DIR = f"{BASE}/runs_final_v1/supervised"

print("modos          :", MODOS)
print("pesos          :", PESOS_BARRIDO)
print("semilla barrido:", SEMILLA_BARRIDO)
print("semillas final :", SEMILLAS_FINAL)
print("salida         :", OUT_ROOT)
print()
print("runs del barrido : %d" % (len(MODOS) * len(PESOS_BARRIDO)))
print("runs del final   : %d (menos los ya hechos en el barrido)" % (len(MODOS) * len(SEMILLAS_FINAL)))
print("tiempo estimado  : ~%.1f h a 18 min por run" % ((len(MODOS) * len(PESOS_BARRIDO)
          + len(MODOS) * (len(SEMILLAS_FINAL) - 1)) * 18 / 60))


---
### Paso 1c - El cuerpo de un run

Una sola función. El bloque de `cfg` es el del supervisado de `runs_final_v1/supervised/`
copiado tal cual, y **lo único que cambia entre runs son `loss_name`, `boundary_mode`,
`boundary_weight` y la semilla**.

Trae la misma lógica de reanudación del notebook 17: si Colab se desconecta a mitad de la
noche, al volver a ejecutar salta los runs que ya terminaron. Un `best_model.pt` sin
`run_summary.txt` es un run cortado y se reentrena desde cero.


In [ ]:
def run_boundary(modo, peso, semilla, verbose=True):
    """Entrena un supervisado UNM con el termino de borde indicado.

    Devuelve el exp_dir. Si el run ya estaba completo no reentrena nada.
    """
    import gc; gc.collect()
    torch.cuda.empty_cache()

    cfg = get_default_config()
    cfg["img_root"]    = BASE
    cfg["msk_root"]    = BASE
    cfg["rotulos_dir"] = ROTULOS

    _EXP_NAME = f"sup_boundary_{modo}_w{peso:g}"
    _SEED     = semilla
    cfg["exp_dir"] = f"{OUT_ROOT}/{_EXP_NAME}/seed_{_SEED}"

    cfg["arch"]      = "unetpp"
    cfg["backbone"]  = "efficientnet-b3"
    cfg["n_classes"] = 1

    cfg["seed"]                 = _SEED
    cfg["use_semi"]             = False
    cfg["use_temp_consistency"] = False
    cfg["lambda_u"]             = 0.0
    cfg["lambda_t"]             = 0.0

    cfg["image_preproc"]  = "base"
    cfg["mask_smoothing"] = "none"
    cfg["use_fixed_crop"] = False
    cfg["target_size"]    = (320, 320)
    cfg["use_pad"]        = True
    cfg["imagenet_norm"]  = False

    cfg["batch_size"]     = 5
    cfg["num_workers"]    = 4
    cfg["drop_last"]      = True
    cfg["num_augmented"]  = 5
    cfg["lr"]             = 1e-3
    cfg["weight_decay"]   = 1e-4
    cfg["epochs"]         = 2000
    cfg["warmup_epochs"]  = 10
    cfg["patience_es"]    = 40
    cfg["eval_threshold"] = 0.5

    cfg["save_preds_vis"] = False
    cfg["run_ruler_eval"] = True

    # --- LA UNICA DIFERENCIA con runs_final_v1/supervised ---
    cfg["loss_name"]              = "bce_dice_boundary"
    cfg["boundary_mode"]          = modo
    cfg["boundary_weight"]        = peso
    cfg["dw_sigma"]               = DW_SIGMA
    cfg["boundary_start_epoch"]   = BOUNDARY_START_EPOCH
    cfg["boundary_warmup_epochs"] = BOUNDARY_WARMUP_EPOCHS

    _exp_dir = cfg["exp_dir"]
    _has_best    = os.path.isfile(os.path.join(_exp_dir, "best_model.pt"))
    _has_metrics = os.path.isfile(os.path.join(_exp_dir, "test_metrics.csv"))
    _has_report  = len(glob.glob(os.path.join(_exp_dir, "*_run_report.json"))) > 0
    _has_summary = os.path.isfile(os.path.join(_exp_dir, "run_summary.txt"))

    _skip_all  = _has_best and _has_metrics and _has_summary
    _eval_only = _has_best and _has_summary and (not _has_metrics or not _has_report)

    if _has_best and not _has_summary:
        print(f"AVISO {_EXP_NAME}/seed_{_SEED}: best_model.pt sin run_summary.txt.")
        print("      El run anterior se corto a medias. Se reentrena desde cero.")

    if _skip_all:
        print(f"Skipping {_EXP_NAME}/seed_{_SEED}: run completo detectado")
        return _exp_dir

    # guardian en frio: la config debe pedir de verdad el termino de borde
    assert cfg["loss_name"] == "bce_dice_boundary", "esta celda exige la perdida de borde"
    assert cfg["boundary_mode"] == modo
    assert cfg["boundary_weight"] == peso
    assert cfg["use_semi"] is False, "el experimento es supervisado puro"

    if verbose:
        print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    loaders = build_dataloaders(cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds)

    # guardian en caliente: el criterion construido tiene que ser el de borde,
    # con el modo y el peso de esta llamada. Si no, abortar antes de gastar GPU.
    from src.losses import build_criterion, BCEPlusDicePlusBoundary
    _crit = build_criterion(cfg)
    assert isinstance(_crit, BCEPlusDicePlusBoundary), (
        f"FALLO: build_criterion devolvio {type(_crit).__name__}, no la perdida de borde.")
    assert _crit.mode == modo and _crit.boundary_weight == peso, (
        "FALLO: el criterion no recibio el modo o el peso de esta celda.")
    print(f"guardian OK: {type(_crit).__name__} modo={_crit.mode} w={_crit.boundary_weight}")

    _t0 = time.time()
    if _eval_only:
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders,
                                      os.path.join(_exp_dir, "best_model.pt"), [])
    else:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(cfg, artifacts["model"], loaders,
                                      artifacts["best_path"], artifacts["history"])
    print(f"[{_EXP_NAME}/seed_{_SEED}] {(time.time()-_t0)/60:.1f} min")
    print(results)
    return _exp_dir


print("run_boundary definida.")


---
## Paso 2 - El barrido (8 runs, ~2,5 h)

Una semilla por combinación. El objetivo no es medir bien, es descartar los pesos que no
hacen nada y los que rompen el entrenamiento.

**El baseline no se reentrena**: `runs_final_v1/supervised/` ya tiene 5 semillas.


In [ ]:
# === PASO 2: barrido de pesos, 1 semilla ===
_fallos = []
for _modo in MODOS:
    for _peso in PESOS_BARRIDO:
        print("=" * 70)
        print(f">>> {_modo}  w={_peso}  seed={SEMILLA_BARRIDO}")
        print("=" * 70)
        try:
            run_boundary(_modo, _peso, SEMILLA_BARRIDO, verbose=False)
        except Exception as e:
            # Un run que revienta no debe llevarse la noche entera por delante.
            print(f"FALLO en {_modo} w={_peso}: {type(e).__name__}: {e}")
            _fallos.append((_modo, _peso, repr(e)))

print()
print("barrido terminado. fallos:", len(_fallos))
for f in _fallos:
    print("  ", f)


---
## Paso 3 - Elegir el peso, en VALIDACIÓN

Lee `hd95_val` de `val_boundary_at_best_epoch` en cada `run_report.json` y se queda con
el mejor peso de cada modo.

**Nunca mira el test.** Si el peso se eligiera por test, la comparación final no valdría
nada, y es el primer sitio donde miraría alguien que quiera romper el resultado.


In [ ]:
# === PASO 3: elegir el peso por VALIDACION (solo lectura) ===
def leer_report(exp_dir):
    _f = glob.glob(os.path.join(exp_dir, "*_run_report.json"))
    if not _f:
        return None
    with open(_f[0], encoding="utf-8") as fh:
        return json.load(fh)


def val_hd95(exp_dir):
    r = leer_report(exp_dir)
    if r is None:
        return None
    v = r.get("val_boundary_at_best_epoch") or {}
    return v.get("hd95_val")


def val_assd(exp_dir):
    r = leer_report(exp_dir)
    if r is None:
        return None
    v = r.get("val_boundary_at_best_epoch") or {}
    return v.get("assd_val")


# referencia del baseline, en validacion y con la MISMA semilla del barrido
_base_seed_dir = f"{BASELINE_DIR}/seed_{SEMILLA_BARRIDO}"
_base_hd95_val = val_hd95(_base_seed_dir)
print("baseline supervisado, seed %d, hd95_val = %s" % (SEMILLA_BARRIDO, "%.3f" % _base_hd95_val if _base_hd95_val else "no encontrado"))
print()

print("%-12s %8s %12s %12s" % ("modo", "peso", "hd95_val", "assd_val"))
print("-" * 48)
MEJOR = {}
for _modo in MODOS:
    _filas = []
    for _peso in PESOS_BARRIDO:
        _d = f"{OUT_ROOT}/sup_boundary_{_modo}_w{_peso:g}/seed_{SEMILLA_BARRIDO}"
        _h, _a = val_hd95(_d), val_assd(_d)
        _filas.append((_peso, _h, _a))
        print("%-12s %8g %12s %12s" % (_modo, _peso,
                 "%.3f" % _h if _h is not None else "-",
                 "%.3f" % _a if _a is not None else "-"))
    _ok = [f for f in _filas if f[1] is not None]
    if _ok:
        MEJOR[_modo] = min(_ok, key=lambda f: f[1])[0]
    print("-" * 48)

print()
print("MEJOR PESO POR MODO (por hd95_val):", MEJOR)
print("Estos son los que el Paso 4 repite con", SEMILLAS_FINAL)


---
## Paso 4 - El mejor peso de cada modo, con 3 semillas (~1,3 h)

La semilla del barrido ya está hecha, así que aquí solo se entrenan las que faltan.


In [ ]:
# === PASO 4: el mejor peso de cada modo con todas las semillas ===
assert MEJOR, "El Paso 3 no encontro ningun run del barrido. Ejecutalo antes."

for _modo, _peso in MEJOR.items():
    for _seed in SEMILLAS_FINAL:
        print("=" * 70)
        print(f">>> FINAL  {_modo}  w={_peso}  seed={_seed}")
        print("=" * 70)
        try:
            run_boundary(_modo, _peso, _seed, verbose=False)
        except Exception as e:
            print(f"FALLO en {_modo} w={_peso} seed={_seed}: {type(e).__name__}: {e}")

print()
print("Paso 4 terminado.")


---
## Paso 5 - El resumen

Compara contra el baseline **calculado de la misma manera y desde los mismos
`run_report.json`**, no contra los números copiados de la tesis. Así la comparación es
interna y consistente aunque la tesis agregue de otro modo.

Lo que hay que mirar es **ASSD y HD95**, no el F1. La hipótesis de Luís es sobre el
contorno; si el F1 no se mueve pero el HD95 baja, el experimento ha encontrado algo.


In [ ]:
# === PASO 5: RESUMEN (solo lectura, no entrena nada) ===
def metricas_test(exp_dir):
    r = leer_report(exp_dir)
    if r is None:
        return None
    t = r.get("test_metrics") or {}
    b = r.get("test_boundary_metrics") or {}
    return {"f1": t.get("sample_mean_f1"),
            "assd": b.get("assd_mean_px"),
            "hd95": b.get("hd95_mean_px")}


def agregar(dirs):
    vals = [metricas_test(d) for d in dirs]
    vals = [v for v in vals if v and v["f1"] is not None]
    if not vals:
        return None
    out = {"n": len(vals)}
    for k in ("f1", "assd", "hd95"):
        xs = [v[k] for v in vals if v[k] is not None]
        out[k] = (statistics.mean(xs),
                  statistics.stdev(xs) if len(xs) > 1 else 0.0) if xs else (None, None)
    return out


_base = agregar(sorted(glob.glob(f"{BASELINE_DIR}/seed_*")))

print("%-34s %3s %16s %16s %16s" % ("configuracion", "n", "F1", "ASSD (px)", "HD95 (px)"))
print("-" * 90)


def linea(nombre, ag):
    if ag is None:
        print("%-34s %3s %16s %16s %16s" % (nombre, "-", "sin runs", "", ""))
        return
    def fmt(t):
        return "%.4f +/- %.4f" % t if t[0] is not None else "-"
    print("%-34s %3d %16s %16s %16s" % (nombre, ag["n"], fmt(ag["f1"]), fmt(ag["assd"]), fmt(ag["hd95"])))


linea("supervisado (baseline)", _base)
print("-" * 90)
for _modo, _peso in (MEJOR or {}).items():
    _dirs = [f"{OUT_ROOT}/sup_boundary_{_modo}_w{_peso:g}/seed_{s}" for s in SEMILLAS_FINAL]
    _ag = agregar(_dirs)
    linea(f"+ borde {_modo} w={_peso:g}", _ag)
    if _ag and _base:
        for _k, _nom in (("f1", "F1"), ("assd", "ASSD"), ("hd95", "HD95")):
            if _ag[_k][0] is not None and _base[_k][0] is not None:
                _d = _ag[_k][0] - _base[_k][0]
                _signo = "mejor" if (_d < 0) != (_k == "f1") else "peor"
                print("      %-6s delta = %+8.4f  (%s)" % (_nom, _d, _signo))
print("-" * 90)
print()
print("Recordatorio: ASSD y HD95 bajan cuando mejoran; el F1 sube.")
print("El resultado, sea cual sea, va al banco de preguntas de defensa.")
print("NO se mete en la tesis sin decidirlo aparte.")
